# SAP Incident Knowledge Assistant — RAG over Structured Excel Data

This notebook builds a Retrieval-Augmented Generation (RAG) application that lets a support
engineer ask natural-language questions against a historical SAP incident log stored in Excel,
and get back a grounded answer with the incident ID, module, priority, resolution steps and the
exact Excel row it came from.

**Stack:** Python · Pandas · LangChain · Google Gemini (`gemini-2.5-flash`) · Google Generative AI
Embeddings (`gemini-embedding-001`) · FAISS · OpenPyXL

**How to run this notebook**
1. Upload `sap_incidents.xlsx` to the Colab session (or keep it next to this notebook).
2. Add your Gemini API key as a Colab secret named `GOOGLE_API_KEY` (or just set the
   environment variable) — see the *Configuration* cell below.
3. Run all cells top to bottom.

**Offline / no-API-key mode:** if no `GOOGLE_API_KEY` is found, the notebook automatically
switches to a **local fallback mode** — a TF-IDF embedding and a small extractive "mock LLM" —
so that the entire pipeline (loading → cleaning → documents → chunking → vector store →
retrieval → filtering → grounded answer) can still be run end-to-end and graded/demoed without
internet access. Every place where the real Gemini API is used is clearly marked, and swapping
back to the real API only requires providing the key — no other code changes.


## 0. Setup — install dependencies

In [ ]:
# Run this once. Safe to re-run.
!pip -q install pandas openpyxl langchain langchain-community langchain-google-genai \
    google-generativeai faiss-cpu scikit-learn


## 0.1 Configuration & API key

We try, in order:
1. A Colab secret called `GOOGLE_API_KEY` (Colab's "Secrets" panel, the recommended way).
2. An environment variable `GOOGLE_API_KEY`.
3. Otherwise we fall back to **offline mode** (see banner printed below).

Get a free Gemini API key at https://aistudio.google.com/app/apikey if you want to run this
against the real Gemini models.


In [ ]:
import os

GOOGLE_API_KEY = None

# 1) Colab secrets
try:
    from google.colab import userdata  # type: ignore
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    pass

# 2) Environment variable
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

OFFLINE_MODE = GOOGLE_API_KEY in (None, "")

if GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

EMBEDDING_MODEL_NAME = "models/gemini-embedding-001"
CHAT_MODEL_NAME = "gemini-2.5-flash"

if OFFLINE_MODE:
    print("=" * 70)
    print("No GOOGLE_API_KEY found -> running in OFFLINE FALLBACK MODE.")
    print("Embeddings  : local TF-IDF vectorizer (demo/testing only)")
    print("LLM         : local extractive 'mock Gemini' (demo/testing only)")
    print("Set a real GOOGLE_API_KEY to switch to gemini-embedding-001 / "
          f"{CHAT_MODEL_NAME}.")
    print("=" * 70)
else:
    print(f"GOOGLE_API_KEY found -> using real Gemini API "
          f"(embeddings={EMBEDDING_MODEL_NAME}, chat={CHAT_MODEL_NAME}).")


## Task 1 — Load and Explore the Excel Data

We load `sap_incidents.xlsx` with Pandas and inspect its shape, types, and distribution across
`sap_module` and `priority` before touching the RAG pipeline.


In [ ]:
import pandas as pd

EXCEL_PATH = "sap_incidents.xlsx"
SHEET_NAME = "Incidents"

raw_df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)

print("Shape:", raw_df.shape)
print("\nColumns:", list(raw_df.columns))
print("\nFirst 5 rows:")
display(raw_df.head())


In [ ]:
print("Number of records:", len(raw_df))

print("\nMissing values per column:")
print(raw_df.isna().sum())

print("\nData types:")
print(raw_df.dtypes)


In [ ]:
print("Incidents by SAP module:")
print(raw_df["sap_module"].value_counts())

print("\nIncidents by priority:")
print(raw_df["priority"].value_counts())


## Task 2 — Clean and Prepare the Data

We defensively clean the dataframe so the RAG pipeline never sees `NaN`, inconsistent casing,
stray whitespace, or non-numeric resolution times — even though this sample dataset is already
clean, the same code will hold up on a messier real export.


In [ ]:
import re


def clean_text(value, default="Not specified"):
    '''Trim whitespace, collapse repeated spaces, replace missing values.'''
    if pd.isna(value):
        return default
    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\x20-\x7E]", "", text)  # drop non-printable/invalid chars
    return text if text else default


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Drop rows that are completely empty
    df = df.dropna(how="all")

    text_columns = [
        "incident_id", "sap_module", "category", "issue_summary",
        "issue_description", "root_cause", "resolution", "owner_team", "status",
    ]
    for col in text_columns:
        if col in df.columns:
            df[col] = df[col].apply(clean_text)

    # Normalise SAP module capitalisation (keep "SAP XYZ" style consistent)
    if "sap_module" in df.columns:
        df["sap_module"] = df["sap_module"].str.upper().str.replace(
            "SAPSUCCESSFACTORS", "SAP SUCCESSFACTORS", regex=False
        )
        df["sap_module"] = df["sap_module"].replace({
            "SAP MM": "SAP MM", "SAP SD": "SAP SD", "SAP HANA": "SAP HANA",
            "SAP BTP": "SAP BTP", "SAP SUCCESSFACTORS": "SAP SuccessFactors",
        })

    # Normalise priority to upper-case P1..P4 style
    if "priority" in df.columns:
        df["priority"] = df["priority"].apply(
            lambda v: clean_text(v).upper().replace(" ", "")
        )

    # Dates -> consistent ISO format
    if "incident_date" in df.columns:
        df["incident_date"] = pd.to_datetime(
            df["incident_date"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")
        df["incident_date"] = df["incident_date"].fillna("Unknown")

    # Resolution time -> numeric, coerce bad values to NaN then fill with column median
    if "resolution_time_hours" in df.columns:
        df["resolution_time_hours"] = pd.to_numeric(
            df["resolution_time_hours"], errors="coerce"
        )
        median_val = df["resolution_time_hours"].median()
        df["resolution_time_hours"] = df["resolution_time_hours"].fillna(median_val).round(2)

    # Status default
    if "status" in df.columns:
        df["status"] = df["status"].replace("Not specified", "Unknown")

    df = df.reset_index(drop=True)
    return df


clean_df = clean_dataframe(raw_df)
print("Clean shape:", clean_df.shape)
display(clean_df.head())
assert clean_df.isna().sum().sum() == 0, "Cleaning left NaNs behind!"
print("\nNo missing values remain after cleaning.")


## Task 3 — Convert Excel Rows into Documents

Every row becomes exactly **one** `langchain_core.documents.Document`. The page content is a
readable, labelled block of text (good for embedding + good for the LLM to read verbatim), and
all structured fields are preserved as metadata for filtering and citation.


In [ ]:
from langchain_core.documents import Document


def row_to_document(row: pd.Series, row_number: int, sheet_name: str, source_name: str) -> Document:
    '''Convert one cleaned incident row into a single LangChain Document.'''
    resolution_hours = row["resolution_time_hours"]
    page_content = (
        f"Incident ID: {row['incident_id']}\n"
        f"Incident Date: {row['incident_date']}\n"
        f"SAP Module: {row['sap_module']}\n"
        f"Priority: {row['priority']}\n"
        f"Category: {row['category']}\n"
        f"Issue Summary: {row['issue_summary']}\n"
        f"Issue Description: {row['issue_description']}\n"
        f"Root Cause: {row['root_cause']}\n"
        f"Resolution: {row['resolution']}\n"
        f"Owner Team: {row['owner_team']}\n"
        f"Resolution Time: {resolution_hours} hours\n"
        f"Status: {row['status']}"
    )

    metadata = {
        # --- required metadata fields (Task 3 spec) ---
        "source_type": "excel_row",
        "source_name": source_name,
        "sheet_name": sheet_name,
        "row_number": row_number,          # 1-indexed Excel row (header = row 1)
        "incident_id": row["incident_id"],
        "sap_module": row["sap_module"],
        "priority": row["priority"],
        "category": row["category"],
        "owner_team": row["owner_team"],
        # --- extra fields kept for convenient citation/answer-formatting ---
        "resolution_time_hours": float(resolution_hours),
        "status": row["status"],
        "issue_summary": row["issue_summary"],
        "root_cause": row["root_cause"],
        "resolution": row["resolution"],
    }
    return Document(page_content=page_content, metadata=metadata)


def dataframe_to_documents(df: pd.DataFrame, sheet_name: str, source_name: str) -> list[Document]:
    docs = []
    for i, row in df.iterrows():
        # +2 because Excel row 1 is the header and pandas index is 0-based
        excel_row_number = i + 2
        docs.append(row_to_document(row, excel_row_number, sheet_name, source_name))
    return docs


incident_documents = dataframe_to_documents(clean_df, SHEET_NAME, EXCEL_PATH)

print(f"Created {len(incident_documents)} documents.\n")
print("Example document (INC-1006):\n")
example_doc = next(d for d in incident_documents if d.metadata["incident_id"] == "INC-1006")
print(example_doc.page_content)
print("\nMetadata:", example_doc.metadata)


## Task 4 — Row-Aware Chunking

Structured records should **not** be run through a generic fixed-size text splitter — doing so
would slice a single incident's root cause away from its resolution and break retrieval. Instead:

* Each incident stays as **one logical chunk** by default.
* Only if a document's content exceeds a length threshold do we split it into
  `INC-XXXX-chunk-N` pieces, and every chunk still carries the full original metadata plus a
  unique `chunk_id`, so a partial match can still be traced back to its incident and Excel row.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

MAX_CHUNK_CHARS = 800  # generous — our incident blocks are short, this rarely triggers

splitter = RecursiveCharacterTextSplitter(
    chunk_size=MAX_CHUNK_CHARS,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " "],
)


def row_aware_chunk(documents: list[Document]) -> list[Document]:
    chunked_docs = []
    for doc in documents:
        incident_id = doc.metadata["incident_id"]
        if len(doc.page_content) <= MAX_CHUNK_CHARS:
            new_meta = dict(doc.metadata)
            new_meta["chunk_id"] = f"{incident_id}-chunk-1"
            chunked_docs.append(Document(page_content=doc.page_content, metadata=new_meta))
        else:
            pieces = splitter.split_text(doc.page_content)
            for idx, piece in enumerate(pieces, start=1):
                new_meta = dict(doc.metadata)
                new_meta["chunk_id"] = f"{incident_id}-chunk-{idx}"
                chunked_docs.append(Document(page_content=piece, metadata=new_meta))
    return chunked_docs


chunked_documents = row_aware_chunk(incident_documents)

print(f"{len(incident_documents)} incidents -> {len(chunked_documents)} chunks "
      f"(1:1 as expected, since all incidents are under {MAX_CHUNK_CHARS} chars).")
print("\nSample chunk_ids:", [d.metadata["chunk_id"] for d in chunked_documents[:3]])


## Task 5 — Generate Embeddings

In **online mode** we use `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")`.
In **offline mode** we substitute a `TfidfVectorizer`-based embedding class that implements the
same LangChain `Embeddings` interface (`embed_documents` / `embed_query`), so every downstream
cell (vector store, retrieval, filtering) is byte-for-byte identical either way.


In [ ]:
from langchain_core.embeddings import Embeddings
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np


class LocalTfidfEmbeddings(Embeddings):
    '''Offline stand-in for GoogleGenerativeAIEmbeddings — demo/testing only.'''

    def __init__(self, corpus: list[str]):
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self._doc_matrix = self.vectorizer.fit_transform(corpus)
        self._dim = self._doc_matrix.shape[1]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        matrix = self.vectorizer.transform(texts)
        return [row.toarray().flatten().tolist() for row in matrix]

    def embed_query(self, text: str) -> list[float]:
        vec = self.vectorizer.transform([text])
        return vec.toarray().flatten().tolist()


chunk_texts = [d.page_content for d in chunked_documents]

if OFFLINE_MODE:
    embeddings = LocalTfidfEmbeddings(corpus=chunk_texts)
else:
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL_NAME)

# Small sanity test: embed a sample question
sample_question = "HANA database performance issue"
sample_vector = embeddings.embed_query(sample_question)
print(f"Embedded sample question -> vector of length {len(sample_vector)}")
print("First 8 dimensions:", np.round(sample_vector[:8], 4))


## Task 6 — Create the Vector Database

We store the chunked incident documents in a FAISS vector index. FAISS keeps the embedding
vectors alongside each document's text and metadata (including `source_type`, `source_name`,
`row_number`, etc.), so every retrieval hit can be traced straight back to the Excel file.


In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunked_documents, embeddings)

print(f"Vector store built with {vector_store.index.ntotal} vectors "
      f"(dimension={vector_store.index.d}).")


## Task 7 — Test Semantic Retrieval

For every query we print the retrieval rank, incident ID, module, priority, Excel row number,
similarity score (when available) and the retrieved text — so a reviewer can eyeball whether
retrieval quality is good enough to answer the question.


In [ ]:
def display_retrieval(query: str, k: int = 3):
    print(f"\n{'=' * 90}\nQuery: {query}\n{'=' * 90}")
    results = vector_store.similarity_search_with_score(query, k=k)
    for rank, (doc, score) in enumerate(results, start=1):
        m = doc.metadata
        print(f"\nRank {rank}")
        print(f"  Incident ID   : {m['incident_id']}")
        print(f"  SAP Module    : {m['sap_module']}")
        print(f"  Priority      : {m['priority']}")
        print(f"  Excel Row     : {m['row_number']}")
        print(f"  Distance/Score: {score:.4f}  (FAISS L2 distance — lower is more similar)")
        print(f"  Text          :\n{doc.page_content}")
    return results


test_queries = [
    "Which incident was related to HANA memory exhaustion?",
    "Find incidents involving SAP BTP connectivity problems.",
    "Which issue was caused by an incorrect pricing procedure?",
    "Show incidents related to employee integration failures.",
]

for q in test_queries:
    display_retrieval(q, k=3)


## Task 8 — Add Metadata Filtering

Semantic search alone can't guarantee "only P1 incidents" or "only the BTP Platform Team" — for
that we combine similarity search with an explicit metadata filter over the structured fields we
preserved in Task 3.


In [ ]:
def filtered_similarity_search(query: str = "", k: int = 10, **filters):
    '''Semantic search, optionally narrowed by exact-match metadata filters.

    Example: filtered_similarity_search("connectivity issue", priority="P1", sap_module="SAP BTP")
    '''
    # Over-fetch, then filter in Python — keeps this independent of any specific
    # vector store's native filter syntax, and works identically for FAISS or Chroma.
    fetch_k = max(k * 5, len(chunked_documents))
    if query:
        candidates = vector_store.similarity_search_with_score(query, k=fetch_k)
    else:
        # No semantic query -> just scan every document with score None
        candidates = [(d, None) for d in chunked_documents]

    def matches(doc):
        for field, value in filters.items():
            if value is None:
                continue
            if str(doc.metadata.get(field, "")).strip().upper() != str(value).strip().upper():
                return False
        return True

    filtered = [(doc, score) for doc, score in candidates if matches(doc)]
    return filtered[:k]


def print_filtered(results):
    if not results:
        print("No matching incidents found for this filter.")
        return
    for rank, (doc, score) in enumerate(results, start=1):
        m = doc.metadata
        score_str = f"{score:.4f}" if score is not None else "n/a"
        print(f"Rank {rank}: {m['incident_id']} | {m['sap_module']} | {m['priority']} "
              f"| row {m['row_number']} | score {score_str} | owner: {m['owner_team']}")


print("Retrieve only P1 incidents:")
print_filtered(filtered_similarity_search(priority="P1", k=10))

print("\nSearch only SAP HANA incidents:")
print_filtered(filtered_similarity_search(sap_module="SAP HANA", k=10))

print("\nFind SAP BTP incidents handled by the BTP Platform Team:")
print_filtered(filtered_similarity_search(sap_module="SAP BTP", owner_team="BTP Platform Team", k=10))


## Task 9 — Create the RAG Prompt

The prompt forces Gemini to answer **only** from retrieved context, name the incident ID,
module and priority, explain the resolution, cite the Excel row, and explicitly say when it
doesn't know — this is the main lever against hallucination.


In [ ]:
RAG_PROMPT_TEMPLATE = '''You are an SAP incident-support assistant. Answer the user's \
question using ONLY the incident records provided in the context below.

Rules you must always follow:
- Never invent an incident, resolution, or detail that is not present in the context.
- Always mention the Incident ID(s) you are basing the answer on.
- Always mention the SAP Module and Priority of each incident you reference.
- Explain the resolution steps that were actually taken, in your own words.
- Always cite the Excel row number(s) the answer came from, in the form: \
Source: {source_name}, Excel row {row_number}.
- If the context does not contain enough information to answer the question, respond with \
exactly this sentence and nothing else: \
"I could not find sufficient information in the incident dataset to answer this question."

Context (retrieved incident records):
{context}

User question: {question}

Answer:'''

print(RAG_PROMPT_TEMPLATE)


## Task 10 — Build the Final RAG Function

`ask_incident_rag(question, top_k=5, sap_module=None, priority=None, owner_team=None)` ties
everything together: filtered/semantic retrieval → context formatting → Gemini call → grounded
answer → printed sources.

In offline mode, `MockGeminiLLM` plays the role of Gemini: it applies the **exact same grounding
rules** (only use retrieved context, cite row numbers, refuse if nothing relevant was retrieved)
using simple text templating instead of a real language model, purely so the full pipeline can be
exercised without an API key.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI


class MockGeminiLLM:
    '''Offline stand-in for ChatGoogleGenerativeAI — demo/testing only.

    Produces a grounded, templated answer strictly from the supplied context documents,
    following the same rules given to the real Gemini prompt (cite incident id / module /
    priority / row number, and an explicit refusal when nothing relevant is found).
    '''

    FALLBACK = "I could not find sufficient information in the incident dataset to answer this question."

    def answer(self, question: str, docs_with_scores) -> str:
        if not docs_with_scores:
            return self.FALLBACK

        # A very small relevance guard for offline TF-IDF mode: if the top hit shares no
        # vocabulary at all with the question, treat it as "not found" rather than guessing.
        top_doc, top_score = docs_with_scores[0]
        if OFFLINE_MODE:
            q_terms = set(w.lower() for w in question.split() if len(w) > 3)
            doc_terms = set(w.lower().strip(".,?") for w in top_doc.page_content.split())
            if not (q_terms & doc_terms):
                return self.FALLBACK

        lines = []
        for doc, _ in docs_with_scores:
            m = doc.metadata
            lines.append(
                f"Incident {m['incident_id']} ({m['sap_module']}, {m['priority']}): "
                f"the issue was \"{m['issue_summary']}\", caused by {m['root_cause']}. "
                f"Resolution: {m['resolution']} (owner: {m['owner_team']}, "
                f"resolution time {m['resolution_time_hours']} hours). "
                f"Source: {m['source_name']}, Excel row {m['row_number']}."
            )
        return "\n\n".join(lines)


if not OFFLINE_MODE:
    gemini_llm = ChatGoogleGenerativeAI(model=CHAT_MODEL_NAME, temperature=0)
else:
    gemini_llm = MockGeminiLLM()


def format_context(docs_with_scores) -> str:
    blocks = []
    for doc, _ in docs_with_scores:
        m = doc.metadata
        blocks.append(
            f"[source_name={m['source_name']}, row_number={m['row_number']}, "
            f"incident_id={m['incident_id']}]\n{doc.page_content}"
        )
    return "\n\n---\n\n".join(blocks)


def ask_incident_rag(question: str, top_k: int = 5, sap_module: str = None,
                      priority: str = None, owner_team: str = None, verbose: bool = True):
    '''Answer a natural-language question about SAP incidents, grounded in sap_incidents.xlsx.'''
    results = filtered_similarity_search(
        query=question, k=top_k,
        sap_module=sap_module, priority=priority, owner_team=owner_team,
    )

    if not results:
        answer = MockGeminiLLM.FALLBACK
    elif OFFLINE_MODE:
        answer = gemini_llm.answer(question, results)
    else:
        context = format_context(results)
        prompt = RAG_PROMPT_TEMPLATE.format(
            context=context, question=question, source_name=EXCEL_PATH,
            row_number="<see context>",
        )
        response = gemini_llm.invoke(prompt)
        answer = response.content

    if verbose:
        print(f"Q: {question}")
        if sap_module or priority or owner_team:
            print(f"   (filters: sap_module={sap_module}, priority={priority}, owner_team={owner_team})")
        print(f"\nA: {answer}\n")
        print("Sources used:")
        for doc, score in results:
            m = doc.metadata
            print(f"  - {m['incident_id']} | {m['sap_module']} | {m['priority']} | "
                  f"row {m['row_number']} | {m['source_name']}")
        print("-" * 90)

    return {"answer": answer, "sources": [d.metadata for d, _ in results]}


# quick smoke test
_ = ask_incident_rag("Which P1 SAP HANA incident took the longest to resolve?", top_k=3,
                      sap_module="SAP HANA", priority="P1")


## Mandatory Test Questions

Running every required question from the problem statement, grouped by category.


In [ ]:
print("### Direct factual questions ###\n")
ask_incident_rag("What was the resolution for incident INC-1004?", top_k=3)
ask_incident_rag("Which team resolved the employee replication issue?", top_k=3)


In [ ]:
print("### Semantic questions ###\n")
ask_incident_rag("Has there been an issue where a cloud application could not connect to an SAP backend?", top_k=3)
ask_incident_rag("Find a previous incident involving database memory problems.", top_k=3)


In [ ]:
print("### Comparison questions ###\n")
ask_incident_rag("Which P1 incident took the longest time to resolve?", top_k=8, priority="P1")
ask_incident_rag("Compare the two SAP HANA P1 incidents.", top_k=8, sap_module="SAP HANA", priority="P1")


In [ ]:
print("### Filter-based questions ###\n")
ask_incident_rag("Show only P1 SAP BTP incidents.", top_k=8, sap_module="SAP BTP", priority="P1")
ask_incident_rag("Find SAP MM incidents handled by the Procure-to-Pay Support team.",
                  top_k=8, sap_module="SAP MM", owner_team="Procure-to-Pay Support")


In [ ]:
print("### Recommendation-style question ###\n")
ask_incident_rag(
    "A user reports that a BTP application cannot access the backend because authentication "
    "is failing. Which previous incident is most similar, and what resolution should the "
    "support team investigate?",
    top_k=3, sap_module="SAP BTP",
)


In [ ]:
print("### Unsupported question (must not invent an answer) ###\n")
result = ask_incident_rag("What is the annual revenue of the company?", top_k=3)
assert "could not find sufficient information" in result["answer"].lower(), (
    "The system attempted to answer an out-of-scope question instead of refusing!"
)
print("PASSED: system correctly refused to answer an out-of-scope question.")


## Design Considerations (Written Explanation)

**1. Why is one Excel row treated as one document?**
Each row is a complete, self-contained incident record — issue, root cause and resolution belong
together as a single unit of meaning. Splitting fields across chunks would let the retriever
return a root cause without its resolution, or vice-versa, producing a technically "relevant"
but practically useless answer.

**2. Why can normal fixed-size chunking damage structured records?**
A generic character/token-based splitter doesn't know where one incident ends and another
begins. It can cut a resolution sentence in half, merge the tail of one incident with the start
of the next, or split a short record for no reason — all of which corrupt the record boundary
that gives the row its meaning.

**3. Why is metadata important in structured-data RAG?**
Metadata (`incident_id`, `sap_module`, `priority`, `owner_team`, `row_number`, `source_name`,
etc.) is what turns free-text retrieval back into something a support engineer can act on and
audit: it enables exact filtering (Task 8), and it gives every answer a verifiable citation back
to the source Excel row, rather than an unattributed paragraph.

**4. How does vector search differ from exact keyword search?**
Keyword search matches literal substrings/tokens, so "backend connectivity failure" would miss a
row that says "application could not connect to backend system" unless the exact words overlap.
Vector search compares meaning via embeddings, so semantically similar but differently-worded
incidents are still retrieved. The trade-off is that vector search can occasionally surface
loosely related results, which is why filtering and a well-grounded prompt matter.

**5. Why combine structured filtering with semantic search?**
Semantic search alone can't reliably guarantee a hard constraint like "only P1" or "only the BTP
Platform Team" — a nearby-but-wrong-priority incident could still rank highly by similarity.
Metadata filtering enforces the exact constraint, while semantic search still ranks by relevance
within that constrained set.

**6. How is hallucination reduced using a grounded prompt?**
The prompt explicitly instructs the model to answer only from retrieved context, to cite the
incident ID and Excel row for every claim, and to output a fixed refusal sentence when the
context is insufficient — removing the model's freedom to "fill in" plausible-sounding details
that were never in the data, and giving reviewers an explicit signal (the refusal string) they
can check for programmatically, as demonstrated in the "unsupported question" test above.

**7. Why may RAG not be suitable for exact aggregation without additional logic?**
RAG retrieves the *top-k most similar* documents — it has no guarantee of seeing *every* row that
matches a criterion, so it cannot reliably compute an average, a count, or a "which one is
highest" answer across the full dataset. Those are exact-aggregation questions, and they need a
calculation layer (Pandas/SQL) that scans the complete table, not a similarity-ranked subset.


## Handling the Aggregation Limitation — Question Router

Analytical questions ("average resolution time", "how many P1 incidents") are answered directly
from the full, clean dataframe with Pandas — not from the vector store — because they require
scanning every row rather than the top-k retrieved ones. A tiny keyword-based classifier decides
which path to take; a production system would likely use an LLM function-call/router instead of
keyword rules, but the architecture is the same.


In [ ]:
import re as _re

ANALYTICAL_HINTS = [
    "average", "avg", "mean", "how many", "count", "total", "sum",
    "highest", "lowest", "most", "least", "maximum", "minimum",
]


def is_analytical_question(question: str) -> bool:
    q = question.lower()
    return any(hint in q for hint in ANALYTICAL_HINTS)


def answer_analytical_question(question: str) -> str:
    q = question.lower()

    # Check the more specific "which X has the highest average" pattern before the
    # generic "average" pattern, since it contains the word "average" too.
    if "highest average" in q or ("highest" in q and "average" in q):
        grouped = clean_df.groupby("sap_module")["resolution_time_hours"].mean().sort_values(ascending=False)
        top_module, top_avg = grouped.index[0], grouped.iloc[0]
        return f"{top_module} has the highest average resolution time: {top_avg:.2f} hours."

    if "average" in q or "avg" in q or "mean" in q:
        module_match = None
        for module in clean_df["sap_module"].unique():
            if module.lower() in q:
                module_match = module
                break
        subset = clean_df if not module_match else clean_df[clean_df["sap_module"] == module_match]
        if subset.empty:
            return "No matching incidents found to compute an average."
        avg_hours = subset["resolution_time_hours"].mean()
        scope = module_match or "all incidents"
        return f"Average resolution time for {scope}: {avg_hours:.2f} hours (n={len(subset)})."

    if "how many" in q or "count" in q:
        priority_match = _re.search(r"\bp[1-4]\b", q)
        if priority_match:
            p = priority_match.group(0).upper()
            n = (clean_df["priority"] == p).sum()
            return f"Number of {p} incidents reported: {n}."
        return f"Total number of incidents reported: {len(clean_df)}."

    return "This looks like an analytical question, but no matching calculation rule was found."


def ask_incident_smart(question: str, **rag_kwargs):
    '''Routes analytical questions to Pandas and everything else to the RAG pipeline.'''
    if is_analytical_question(question):
        print(f"Q: {question}\n[routed to: ANALYTICAL / Pandas]")
        answer = answer_analytical_question(question)
        print(f"A: {answer}\n" + "-" * 90)
        return {"answer": answer, "route": "analytical"}
    else:
        print(f"[routed to: SEMANTIC / Vector RAG]")
        result = ask_incident_rag(question, **rag_kwargs)
        result["route"] = "semantic"
        return result


# Demonstrate the limitation and its fix
ask_incident_smart("What is the average resolution time for SAP HANA incidents?")
ask_incident_smart("How many P1 incidents were reported?")
ask_incident_smart("Which module has the highest average resolution time?")


## Summary & Limitations

**What this notebook delivers**
- Clean, validated incident data loaded from `sap_incidents.xlsx`.
- One LangChain `Document` per incident row, with full metadata for traceability.
- Row-aware chunking that only splits when a record is genuinely long, preserving meaning.
- Embeddings + a FAISS vector store enabling semantic retrieval over incident text.
- Metadata filtering combinable with semantic search (`priority`, `sap_module`, `owner_team`).
- A grounded prompt and `ask_incident_rag()` function that cites the incident ID, module,
  priority and Excel row for every answer, and explicitly refuses out-of-scope questions.
- A lightweight router (`ask_incident_smart()`) that sends aggregation-style questions to Pandas
  instead of the vector store, since RAG's top-k retrieval cannot guarantee full-table coverage.

**Known limitations / future improvements**
- The offline fallback (TF-IDF + templated answers) is for demonstration only; retrieval quality
  and answer fluency with the real `gemini-embedding-001` + `gemini-2.5-flash` will be
  meaningfully better, especially on paraphrased or vague questions.
- The analytical router uses simple keyword rules; a production system should use an LLM-based
  classifier or function-calling to route more reliably.
- With only 8 sample incidents, retrieval quality can't be stress-tested at scale — before
  production use, this should be validated against a dataset with thousands of rows.
- Bonus directions worth pursuing: hybrid keyword + vector search, similarity-score thresholds
  to trigger the refusal path even when *some* documents are retrieved, conversation memory for
  follow-up questions, and a Streamlit front end for support engineers.
